<a href="https://colab.research.google.com/github/aljeralmazan28-web/Algoverse-Research-TNAM/blob/main/moli_latefusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

MOLI_ZIP = '/content/drive/MyDrive/MOLI.zip'        # edit
OUT_DIR  = '/content/drive/MyDrive/moli_results'    # splits + predictions are saved here (persists across sessions)
SEX_CSV  = None                                     # later: path to sex.csv (cell line ID -> F/M)
DRUG     = 'Paclitaxel'
DATA_DIR = '/content/moli_data'

import os, sys, subprocess
def sh(cmd):
    r = subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True)
    return r.stdout

os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
sh(f'unzip -q -o "{MOLI_ZIP}" -d /content')
sys.path.insert(0, '/content/MOLI')                 # utils.py (AllTripletSelector)

# Zenodo record 4036592: expression + response (already used for baseline 1), plus binarized CNA and mutations (SNA)
BASE = 'https://zenodo.org/records/4036592/files'
for name in ['exprs_homogenized', 'response', 'CNA_binary', 'SNA_binary']:
    marker = f'{DATA_DIR}/.done_{name}'
    if not os.path.exists(marker):
        tar = f'{DATA_DIR}/{name}.tar.gz'
        if not os.path.exists(tar):
            sh(f'wget -q -O {tar}.part "{BASE}/{name}.tar.gz?download=1" && mv {tar}.part {tar}')
        sh(f'tar -xzf {tar} -C {DATA_DIR}')
        open(marker, 'w').close()
print(sh(f'find {DATA_DIR} -name "*{DRUG}*" | sort'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/moli_data/CNA_binary/CCLE_CNA.Paclitaxel.tsv
/content/moli_data/CNA_binary/GDSC_CNA.Paclitaxel.tsv
/content/moli_data/CNA_binary/PDX_CNA.Paclitaxel.tsv
/content/moli_data/CNA_binary/TCGA_CNA.Paclitaxel.tsv
/content/moli_data/exprs_homogenized/GDSC_exprs.Paclitaxel.eb_with.GSE25065_exprs.Paclitaxel.tsv
/content/moli_data/exprs_homogenized/GDSC_exprs.Paclitaxel.eb_with.PDX_exprs.Paclitaxel.tsv
/content/moli_data/exprs_homogenized/GDSC_exprs.Paclitaxel.eb_with.TCGA_exprs.Paclitaxel.tsv
/content/moli_data/exprs_homogenized/PDX_exprs.Paclitaxel.eb_with.GDSC_exprs.Paclitaxel.tsv
/content/moli_data/exprs_homogenized/TCGA_exprs.Paclitaxel.eb_with.GDSC_exprs.Paclitaxel.tsv
/content/moli_data/response/CCLE_response.Paclitaxel.tsv
/content/moli_data/response/GDSC_response.Paclitaxel.tsv
/content/moli_data/response/GSE25065_response.Paclitaxel.tsv
/content/moli_

In [3]:
import json, random
from glob import glob
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from utils import AllTripletSelector

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [4]:
def find(pattern):
    hits = sorted(glob(f"{DATA_DIR}/**/{pattern}", recursive=True))
    return hits

e_files = [f for f in find(f"GDSC_exprs.{DRUG}.eb_with.*") if '.z.' not in f]
e_files = sorted(e_files, key=lambda f: 'PDX' not in f)       # prefer the PDX-harmonized file, as in baseline 1
m_files = find(f"GDSC_mutations.{DRUG}.tsv")
c_files = find(f"GDSC_CNA.{DRUG}.tsv")
r_files = find(f"GDSC_response.{DRUG}.tsv")
for k, v in dict(expression=e_files, mutations=m_files, CNA=c_files, response=r_files).items():
    print(k, v)
assert e_files and m_files and c_files and r_files, "A file was not found: check the file list printed in the setup cell."

def load_omic(path):
    df = pd.read_csv(path, sep="\t", index_col=0, decimal=".").T   # rows = cell lines, cols = genes
    df.index = df.index.astype(str); df.columns = df.columns.astype(str)
    return df

E = pd.read_csv(e_files[0], sep="\t", index_col=0, decimal=",").T
E.index = E.index.astype(str); E.columns = E.columns.astype(str)
E = E.dropna(axis=1)
M, C = load_omic(m_files[0]), load_omic(c_files[0])
M = M.fillna(0); M = (M != 0).astype(np.float32)       # binarize, as in the original script
C = C.fillna(0); C = (C != 0).astype(np.float32)
C = C.loc[:, ~C.columns.duplicated()]
M = M.loc[:, ~M.columns.duplicated()]
E = E.loc[:, ~E.columns.duplicated()]

R = pd.read_csv(r_files[0], sep="\t", index_col=0, decimal=",")
R.index = R.index.astype(str)
resp = R['response'] if 'response' in R.columns else R.iloc[:, 0]
y_all = resp.map({'R': 0, 'S': 1}).dropna().astype(int)

# common genes (same columns for all three omics, as in the original) and common cell lines
gM, gC, iM, iC, iY = set(M.columns), set(C.columns), set(M.index), set(C.index), set(y_all.index)
genes = [g for g in E.columns if g in gM and g in gC]
ids = [i for i in E.index if i in iM and i in iC and i in iY]
E, M, C = E.loc[ids, genes], M.loc[ids, genes], C.loc[ids, genes]
y_all = y_all.loc[ids]
D = {'E': E, 'M': M, 'C': C}
print(f"{len(ids)} cell lines, {len(genes)} genes | sensitive: {y_all.sum()}, resistant: {(y_all == 0).sum()}")
print("mutation density:", round(float(M.values.mean()), 4), "| CNA density:", round(float(C.values.mean()), 4))

expression ['/content/moli_data/exprs_homogenized/GDSC_exprs.Paclitaxel.eb_with.PDX_exprs.Paclitaxel.tsv', '/content/moli_data/exprs_homogenized/GDSC_exprs.Paclitaxel.eb_with.GSE25065_exprs.Paclitaxel.tsv', '/content/moli_data/exprs_homogenized/GDSC_exprs.Paclitaxel.eb_with.TCGA_exprs.Paclitaxel.tsv']
mutations ['/content/moli_data/SNA_binary/GDSC_mutations.Paclitaxel.tsv']
CNA ['/content/moli_data/CNA_binary/GDSC_CNA.Paclitaxel.tsv']
response ['/content/moli_data/response/GDSC_response.Paclitaxel.tsv']
389 cell lines, 17337 genes | sensitive: 26, resistant: 363
mutation density: 0.0228 | CNA density: 0.3469


In [5]:
SPLITS_PATH = f"{OUT_DIR}/splits_{DRUG}.json"
ids_arr = np.array(ids)

if os.path.exists(SPLITS_PATH):
    splits = json.load(open(SPLITS_PATH))
    idset = set(ids)
    splits = [{k: [str(i) for i in v if str(i) in idset] for k, v in s.items()} for s in splits]
    print("Loaded existing splits:", SPLITS_PATH)
else:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    splits = [{'train': ids_arr[tr].tolist(), 'test': ids_arr[te].tolist()}
              for tr, te in skf.split(ids_arr, y_all.values)]
    json.dump(splits, open(SPLITS_PATH, 'w'))
    print("Created and saved splits:", SPLITS_PATH)

for k, s in enumerate(splits, 1):
    yt = y_all.loc[s['test']]
    print(f"fold {k}: train={len(s['train'])} test={len(s['test'])} (test S={yt.sum()}, R={(yt == 0).sum()})")

Created and saved splits: /content/drive/MyDrive/moli_results/splits_Paclitaxel.json
fold 1: train=311 test=78 (test S=5, R=73)
fold 2: train=311 test=78 (test S=5, R=73)
fold 3: train=311 test=78 (test S=5, R=73)
fold 4: train=311 test=78 (test S=6, R=72)
fold 5: train=312 test=77 (test S=5, R=72)


In [6]:
class MOLI(nn.Module):
    def __init__(self, in_dims, h_dims, enc_rates, clf_rate, fusion_norm='dim0'):
        super().__init__()
        self.encs = nn.ModuleList([
            nn.Sequential(nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(r))
            for d, h, r in zip(in_dims, h_dims, enc_rates)])
        self.clf = nn.Sequential(nn.Linear(sum(h_dims), 1), nn.Dropout(clf_rate), nn.Sigmoid())
        self.fusion_norm = fusion_norm
    def embed(self, xs):
        z = torch.cat([enc(x) for enc, x in zip(self.encs, xs)], dim=1)
        if self.fusion_norm == 'dim0':   z = F.normalize(z, p=2, dim=0)
        elif self.fusion_norm == 'row':  z = F.normalize(z, p=2, dim=1)
        return z
    def forward(self, xs):
        z = self.embed(xs)
        return self.clf(z), z

triplet_selector = AllTripletSelector()
T = lambda a: torch.tensor(a, dtype=torch.float32)

def train_fold(Xtr, ytr, Xte, p, omics, seed=SEED):
    # Xtr / Xte: {'E': array, 'M': array, 'C': array}, all with the same gene columns
    torch.manual_seed(seed); np.random.seed(seed)

    # variance filter fit on the training fold's expression only; the same genes are kept for every omic (as in the original)
    keep = VarianceThreshold(0.05).fit(Xtr['E']).get_support()
    Xtr = {o: Xtr[o][:, keep] for o in omics}
    Xte = {o: Xte[o][:, keep] for o in omics}
    sc = StandardScaler().fit(Xtr['E'])                     # only expression is standardized; M and C stay binary
    Xtr['E'], Xte['E'] = sc.transform(Xtr['E']), sc.transform(Xte['E'])

    counts = np.array([(ytr == c).sum() for c in np.unique(ytr)])
    w = (1.0 / counts)[ytr]
    sampler = WeightedRandomSampler(torch.from_numpy(w).double(), len(w), replacement=True)
    loader = DataLoader(TensorDataset(*[T(Xtr[o]) for o in omics], T(ytr)),
                        batch_size=p['mb_size'], sampler=sampler, drop_last=True)   # drop_last: BatchNorm needs >1 sample

    model = MOLI([Xtr[o].shape[1] for o in omics], p['h_dims'], p['enc_rates'], p['clf_rate'], p['fusion_norm'])
    groups = [{'params': enc.parameters(), 'lr': lr} for enc, lr in zip(model.encs, p['lrs'])]
    groups.append({'params': model.clf.parameters(), 'lr': p['lr_clf'], 'weight_decay': p['wd']})
    opt = optim.Adagrad(groups)
    trip = nn.TripletMarginLoss(margin=p['marg'], p=2)
    bce = nn.BCELoss()

    for epoch in range(p['epochs']):
        model.train()
        for *xb, yb in loader:
            if yb.mean() in (0., 1.):
                continue
            pred, z = model(xb)
            loss = bce(pred, yb.view(-1, 1))
            t = triplet_selector.get_triplets(z, yb)
            if t.dim() == 2 and len(t) > 0:
                loss = loss + p['lam'] * trip(z[t[:, 0]], z[t[:, 1]], z[t[:, 2]])
            opt.zero_grad(); loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
        return model([T(Xte[o]) for o in omics])[0].numpy().ravel()

In [7]:
def run_cv(p, omics, name, splits=splits, save=True):
    rows, aucs = [], []
    for k, s in enumerate(splits, 1):
        Xtr = {o: D[o].loc[s['train']].values.astype(np.float32) for o in omics}
        Xte = {o: D[o].loc[s['test']].values.astype(np.float32) for o in omics}
        ytr, yte = y_all.loc[s['train']].values, y_all.loc[s['test']].values
        pred = train_fold(Xtr, ytr, Xte, p, omics)
        aucs.append(roc_auc_score(yte, pred))
        print(f"  fold {k}: AUROC={aucs[-1]:.3f}")
        rows.append(pd.DataFrame({'id': s['test'], 'fold': k, 'y': yte, 'pred': pred}))
    oof = pd.concat(rows, ignore_index=True)
    if save: oof.to_csv(f"{OUT_DIR}/oof_{name}_{DRUG}.csv", index=False)
    print(f"{name}: mean AUROC {np.mean(aucs):.3f} +- {np.std(aucs):.3f} | "
          f"pooled AUPRC {average_precision_score(oof['y'], oof['pred']):.3f}")
    return oof, aucs

# starting point drawn from the original search grids; tune before reporting
moli_params = dict(mb_size=36, h_dims=[256, 128, 128], enc_rates=[0.4, 0.4, 0.4], clf_rate=0.4,
                   marg=1.0, lrs=[0.01, 0.01, 0.01], lr_clf=0.01, wd=0.001, epochs=50, lam=0.2,
                   fusion_norm='dim0')

# smoke test first (3 epochs; AUROC will be rough but nothing should crash)
_ = run_cv({**moli_params, 'epochs': 3}, ['E', 'M', 'C'], 'smoke', save=False)

  fold 1: AUROC=0.721
  fold 2: AUROC=0.674
  fold 3: AUROC=0.784
  fold 4: AUROC=0.569
  fold 5: AUROC=0.892
smoke: mean AUROC 0.728 +- 0.108 | pooled AUPRC 0.183


In [8]:
oof_moli, aucs_moli = run_cv(moli_params, ['E', 'M', 'C'], 'moli_latefusion')

  fold 1: AUROC=0.745
  fold 2: AUROC=0.578
  fold 3: AUROC=0.740
  fold 4: AUROC=0.553
  fold 5: AUROC=0.858
moli_latefusion: mean AUROC 0.695 +- 0.114 | pooled AUPRC 0.133


In [ ]:
b1_params = dict(mb_size=36, h_dims=[256], enc_rates=[0.5], clf_rate=0.5, marg=1.0, lrs=[0.01],
                 lr_clf=0.01, wd=0.001, epochs=50, lam=0.1, fusion_norm=None)
oof_b1, aucs_b1 = run_cv(b1_params, ['E'], 'moli_exprs_only') # rerun baseline 1

  fold 1: AUROC=0.627
  fold 2: AUROC=0.647


In [ ]:
sex = pd.read_csv(SEX_CSV, index_col=0).iloc[:, 0] if SEX_CSV else None

def sex_report(oof, name, n_boot=1000):
    sx = sex.reindex(oof['id'].astype(str)).values
    yt, pr = oof['y'].values, oof['pred'].values
    def auroc(m): return roc_auc_score(yt[m], pr[m]) if len(set(yt[m])) == 2 else np.nan
    out = {g: auroc(sx == g) for g in ['F', 'M']}
    rng = np.random.default_rng(0); gaps = []
    f_idx, m_idx = np.where(sx == 'F')[0], np.where(sx == 'M')[0]
    for _ in range(n_boot):
        fb, mb = rng.choice(f_idx, len(f_idx)), rng.choice(m_idx, len(m_idx))
        if len(set(yt[fb])) == 2 and len(set(yt[mb])) == 2:
            gaps.append(roc_auc_score(yt[fb], pr[fb]) - roc_auc_score(yt[mb], pr[mb]))
    print(f"{name}: F n={len(f_idx)} AUROC={out['F']:.3f} | M n={len(m_idx)} AUROC={out['M']:.3f} | "
          f"gap F-M={np.mean(gaps):.3f} 95% CI [{np.percentile(gaps, 2.5):.3f}, {np.percentile(gaps, 97.5):.3f}]")

if sex is not None:
    sex.index = sex.index.astype(str)
    sex_report(oof_b1, 'baseline 1 (expression only)')
    sex_report(oof_moli, 'baseline 2 (MOLI late fusion)')
else:
    print("Set SEX_CSV in the setup cell to run this.")